# agent_framework_azure_cosmos

## Example 1: a chat-history provider

In [ ]:
# Example 1: CosmosHistoryProvider keeps the database and container it was given
from agent_framework_azure_cosmos import CosmosHistoryProvider

history = CosmosHistoryProvider(endpoint="https://example.documents.azure.com:443/", database_name="agents", container_name="history", credential="test-key")
print("Example 1:", type(history).__name__, history.source_id)
assert history.source_id == "azure_cosmos_history"

## Example 2: what the history provider stores is configurable

In [ ]:
# Example 2: store inputs only, and do not load messages back into the prompt
from agent_framework_azure_cosmos import CosmosHistoryProvider

history = CosmosHistoryProvider("audit_log", load_messages=False, store_outputs=False, endpoint="https://example.documents.azure.com:443/",
                                database_name="agents", container_name="audit", credential="test-key")
assert history.source_id == "audit_log" and history.load_messages is False and history.store_outputs is False
print("Example 2:", history.source_id, "load_messages", history.load_messages, "store_outputs", history.store_outputs)

## Example 3: workflow checkpoint storage

In [ ]:
# Example 3: CosmosCheckpointStorage is the Cosmos DB backend for workflow checkpoints
from agent_framework_azure_cosmos import CosmosCheckpointStorage

storage = CosmosCheckpointStorage(endpoint="https://example.documents.azure.com:443/", database_name="agents", container_name="checkpoints", credential="test-key")
methods = sorted(m for m in dir(storage) if not m.startswith("_") and callable(getattr(storage, m)))
assert any("checkpoint" in m or m in ("save", "load") for m in methods), methods  # the CheckpointStorage protocol's calls
print("Example 3:", type(storage).__name__, methods[:6])

## Example 4: settings from AZURE_COSMOS_* environment variables

In [ ]:
# Example 4: with no endpoint argument the provider reads its connection settings from the environment
import os
from unittest import mock
from agent_framework_azure_cosmos import CosmosHistoryProvider

env = {"AZURE_COSMOS_ENDPOINT": "https://from-env.documents.azure.com:443/", "AZURE_COSMOS_DATABASE_NAME": "envdb",
       "AZURE_COSMOS_CONTAINER_NAME": "envc", "AZURE_COSMOS_KEY": "k"}
with mock.patch.dict(os.environ, env):
    history = CosmosHistoryProvider()
print("Example 4: built from the environment:", type(history).__name__)

## Example 5: `agent_framework.azure` re-exports this package

In [ ]:
# Example 5: the umbrella namespace for Azure connectors hands back the same classes
import importlib.metadata
import agent_framework.azure as via_core
import agent_framework_azure_cosmos as direct

assert via_core.CosmosHistoryProvider is direct.CosmosHistoryProvider
print("Example 5:", importlib.metadata.version("agent-framework-azure-cosmos"), "re-exported through agent_framework.azure")